# 01 | Business Understanding

## Objetivo del notebook

Este notebook explica el problema de negocio antes de entrenar cualquier modelo. La idea es que una persona que no conoce el proyecto pueda entender:

1. Qué decisión quiere apoyar el modelo.
2. Qué significa la variable objetivo.
3. Qué error es más costoso para la empresa.
4. Qué métricas y criterios se usarán para seleccionar el modelo final.

Este notebook corresponde a la fase **Business Understanding** de CRISP-DM y conecta directamente con la evaluación económica que se usa al final del proyecto.


In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)


Project root: /Users/alexandralozano/dp261-g1-final 2


## 1. Problema de negocio

La empresa compra vehículos usados en subastas. Algunas compras terminan siendo malas compras porque el vehículo presenta defectos, costos de reparación o pérdidas comerciales posteriores.

El modelo busca responder una pregunta práctica:

> **Antes de comprar un vehículo, ¿este auto tiene alto riesgo de convertirse en una mala compra?**

La salida no debe ser solo una probabilidad técnica. Para un usuario comercial, la salida debe convertirse en una recomendación simple:

- **Rojo**: alto riesgo, detener compra o pasar a revisión experta.
- **Ámbar**: riesgo medio, revisar manualmente.
- **Verde**: riesgo bajo, continuar evaluación comercial.

Por eso el proyecto no termina en el entrenamiento del modelo; también incluye API, dashboard y una función de valor económico.


## 2. Variable objetivo

La variable objetivo del dataset es `IsBadBuy`.

| Valor | Significado de negocio | Interpretación para el modelo |
|---|---|---|
| `0` | Buen vehículo / compra aceptable | Clase negativa |
| `1` | Mala compra / vehículo riesgoso | Clase positiva |

La clase positiva es la más importante para el negocio porque representa el evento que queremos detectar: comprar un vehículo defectuoso.


In [2]:
import pandas as pd
from src.config import RAW_DATA_PATH, TARGET

df = pd.read_csv(RAW_DATA_PATH)
print(df.shape)
df[TARGET].value_counts(normalize=True).rename('proporcion')


(72983, 33)


IsBadBuy
0    0.877012
1    0.122988
Name: proporcion, dtype: float64

## 3. Qué error duele más

La matriz de confusión se interpreta así:

| Cuadrante | Caso | Decisión | Impacto |
|---|---|---|---|
| TP | El auto era Bad Buy y el modelo lo detecta | No comprar / revisar | Se evita una pérdida |
| TN | El auto era bueno y el modelo lo aprueba | Continuar compra | Se captura margen |
| FP | El auto era bueno pero el modelo lo rechaza | Se pierde oportunidad | Costo comercial moderado |
| FN | El auto era Bad Buy pero el modelo lo aprueba | Se compra un auto malo | Costo más alto |

La observación del profesor fue correcta: **el FN no puede pesar cero**. En este problema, el FN es el peor error porque implica comprar un vehículo defectuoso creyendo que era bueno.


## 4. Función económica final

La selección final del modelo usa una función de negocio que convierte la matriz de confusión en valor monetario:

```text
Valor = TP*2500 + TN*600 + FP*(-900) + FN*(-4500)
```

### Justificación de coeficientes

| Coeficiente | Valor | Justificación |
|---|---:|---|
| TP | +2500 | Ahorro estimado por evitar una mala compra, incluyendo reparación, garantía y pérdida de margen. |
| TN | +600 | Margen operativo aproximado de aprobar correctamente un vehículo bueno. |
| FP | -900 | Costo de oportunidad por rechazar un vehículo que sí podía comprarse. |
| FN | -4500 | Penalización alta por comprar un vehículo defectuoso no detectado. Incluye costo de adquisición, reparación, garantía y daño reputacional. |

Estos valores no pretenden ser contabilidad exacta; son supuestos razonables para comparar modelos de forma consistente. En producción deberían validarse con el sponsor del negocio.


In [3]:
from src.config import BENEFIT_TP, BENEFIT_TN, COST_FP, COST_FN, BUSINESS_THRESHOLD

print({
    'BENEFIT_TP': BENEFIT_TP,
    'BENEFIT_TN': BENEFIT_TN,
    'COST_FP': COST_FP,
    'COST_FN': COST_FN,
    'THRESHOLD': BUSINESS_THRESHOLD,
})


{'BENEFIT_TP': 2500, 'BENEFIT_TN': 600, 'COST_FP': -900, 'COST_FN': -4500, 'THRESHOLD': 0.5}


## 5. Métricas técnicas y criterio de selección

No se usará accuracy como métrica principal porque el dataset está desbalanceado. Un modelo que predice casi todo como “buen vehículo” puede verse bien en accuracy, pero fallar en detectar Bad Buys.

Se monitorean estas métricas:

| Métrica | Por qué importa |
|---|---|
| Recall clase 1 | Mide cuántos Bad Buys detectamos. Es clave porque FN es caro. |
| Precision clase 1 | Evita rechazar demasiados vehículos buenos. |
| F2 | Da más peso al recall que a la precision. Sirve cuando FN cuesta más que FP. |
| ROC-AUC | Mide capacidad general de discriminación. |
| Average Precision | Útil en problemas desbalanceados. |
| Business value | Criterio final para escoger el modelo. |

La decisión final se toma por **valor de negocio**, con threshold estándar `0.5`, y se valida una sola vez sobre el test holdout.


## 6. Entregables conectados al negocio

El proyecto produce:

- Notebooks documentados y secuenciales.
- Pipeline completo de preprocesamiento + feature engineering + modelo.
- Ranking de modelos por métricas técnicas y valor económico.
- Modelo final serializado en `models/final_model.pkl`.
- API que recibe columnas crudas y aplica el pipeline completo.
- Dashboard Streamlit orientado al usuario comercial.
- Contratos de entrada/salida para deployment.
- MLflow Tracking cuando está instalado en el entorno.
